# 📊 Phân tích dữ liệu & Rò rỉ dữ liệu Vietnamese Fake News Detection

Notebook này phân tích toàn diện bộ dữ liệu phát hiện tin giả tiếng Việt, kết hợp dữ liệu gốc và dữ liệu bên ngoài, bao gồm:

0. Cài đặt & Import thư viện
1. Tải & Chuẩn hóa dữ liệu đa nguồn
2. Tổng quan dữ liệu & giá trị thiếu
3. Phân bố nhãn Real vs Fake
4. Phân tích độ dài bài viết
5. Chỉ số tương tác (engagement)
6. Phân tích thời gian & Đặc trưng bài viết
7. Phân tích user (Chuyên sâu: Rò rỉ dữ liệu - Data Leakage)
8. Ma trận tương quan
9. Phân tích hình ảnh
10. Kết luận & Khuyến nghị cho Model

## 0. Cài đặt & Import thư viện
Phần này thiết lập không gian làm việc. Chúng ta sử dụng `seaborn` và `matplotlib` để trực quan hóa, đồng thời tự động tạo một thư mục `charts/` để xuất toàn bộ biểu đồ thành file ảnh.


In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q pandas numpy matplotlib seaborn wordcloud

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Thiết lập style biểu đồ chuẩn học thuật
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Tạo thư mục lưu ảnh
CHART_DIR = "charts/"
os.makedirs(CHART_DIR, exist_ok=True)
print(f"✅ Đã thiết lập thư mục lưu ảnh tại: {CHART_DIR}")

✅ Đã thiết lập thư mục lưu ảnh tại: charts/


## 1. Tải & Chuẩn hóa dữ liệu đa nguồn
**Giải thích chi tiết:** Để đánh giá mô hình AI một cách khách quan, ta không thể chỉ phụ thuộc vào 1 bộ dữ liệu. Ở đây, ta kết hợp bộ dữ liệu gốc với 2 bộ dữ liệu bên ngoài (ViFN và VFND). 
* Việc dùng `pd.concat` giúp gộp các tập train/test bị chia nhỏ của dữ liệu ngoài thành một tập hoàn chỉnh.
* Chuẩn hóa nhãn (0 = Tin Thật, 1 = Tin Giả) là bước bắt buộc để các phân tích so sánh chéo có ý nghĩa.

In [ ]:
# 1. Load dữ liệu gốc
df_original = pd.read_csv('../data/raw/data.csv')

# 2. Load và gộp dữ liệu ViFN (huynhtuan0106)
df_tuan_train = pd.read_csv('../data/external/huynhtuan0106/train_data.csv')
df_tuan_test = pd.read_csv('../data/external/huynhtuan0106/test_data.csv')
df_tuan = pd.concat([df_tuan_train, df_tuan_test], ignore_index=True)

# 3. Load và gộp dữ liệu VFND
df_vnfd_1 = pd.read_csv('../data/external/vnfd/vn_news_223_tdlfr.csv')
df_vnfd_2 = pd.read_csv('../data/external/vnfd/vn_news_226_tlfr.csv')
df_vnfd = pd.concat([df_vnfd_1, df_vnfd_2], ignore_index=True)

# Chuẩn hóa tên cột: Chuyển cột chứa nội dung thành 'text' và cột nhãn thành 'label'
# (Lưu ý: Giả định các bộ dữ liệu ngoài có cột văn bản và nhãn. Code dưới đây xử lý an toàn)
def standardize_columns(df):
    # Tìm cột nhãn (thường là label, class, category)
    label_col = [c for c in df.columns if 'label' in c.lower() or 'class' in c.lower()][0]
    # Tìm cột text (thường là text, content, article, news)
    text_col = [c for c in df.columns if 'text' in c.lower() or 'content' in c.lower() or 'news' in c.lower()][0]
    
    df_clean = df.rename(columns={text_col: 'text', label_col: 'label'})
    return df_clean[['text', 'label']]

df_tuan = standardize_columns(df_tuan)
df_vnfd = standardize_columns(df_vnfd)

print("✅ Kích thước tập Gốc:", df_original.shape)
print("✅ Kích thước tập ViFN:", df_tuan.shape)
print("✅ Kích thước tập VFND:", df_vnfd.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../data/external/vfnd/vn_news_223_tdlfr.csv'

## 2. Tổng quan dữ liệu & giá trị thiếu
Kiểm tra cấu trúc (kiểu dữ liệu) và mức độ sạch của dữ liệu. Nếu cột văn bản (`text`) bị rỗng (NaN), mô hình xử lý ngôn ngữ tự nhiên (NLP) sẽ báo lỗi.

In [ ]:
print("--- Dữ liệu Gốc ---")
print(df_original.isnull().sum()[df_original.isnull().sum() > 0]) # Chỉ in các cột có NaN

print("\n--- Dữ liệu ViFN ---")
print(f"Số giá trị NaN trong text: {df_tuan['text'].isnull().sum()}")

print("\n--- Dữ liệu VFND ---")
print(f"Số giá trị NaN trong text: {df_vnfd['text'].isnull().sum()}")

# Xử lý nhanh giá trị thiếu cho text
df_original['text'] = df_original['text'].fillna('')
df_tuan['text'] = df_tuan['text'].fillna('')
df_vnfd['text'] = df_vnfd['text'].fillna('')

## 3. Phân bố nhãn Real vs Fake
Mất cân bằng lớp (Class Imbalance) là một "căn bệnh" phổ biến trong AI. Nếu dữ liệu có quá nhiều Tin Thật, mô hình sẽ có xu hướng "lười biếng" và luôn dự đoán là Tin Thật để đạt độ chính xác (Accuracy) cao giả tạo. Biểu đồ dưới đây giúp ta so sánh mức độ mất cân bằng giữa các nguồn.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df_original, x='label', ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Tập Gốc (Original)')
axes[0].set_xticklabels(['0 (Thật)', '1 (Giả)'])

sns.countplot(data=df_tuan, x='label', ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Tập ViFN')
axes[1].set_xticklabels(['0 (Thật)', '1 (Giả)'])

sns.countplot(data=df_vnfd, x='label', ax=axes[2], palette=['#2ecc71', '#e74c3c'])
axes[2].set_title('Tập VFND')
axes[2].set_xticklabels(['0 (Thật)', '1 (Giả)'])

plt.suptitle('So sánh phân phối nhãn (Mất cân bằng lớp)', fontsize=16, y=1.05)
plt.tight_layout()
plt.savefig(CHART_DIR + '03_class_imbalance.png', bbox_inches='tight')
plt.show()

## 4. Phân tích độ dài bài viết
**Giải thích chi tiết:** Có một giả thuyết tâm lý học là: "Tin giả thường dài dòng để cố gắng thuyết phục người đọc, hoặc cực kỳ ngắn gọn (giật tít) để gây sốc". 
Ở đây ta dùng biểu đồ phân phối mật độ kết hợp trục Logarit (do độ dài văn bản có nhiều giá trị ngoại lai - Outliers) để xem tin giả và tin thật có khác biệt về số lượng từ hay không.

In [ ]:
# Tính số lượng từ (Word Count) cho cả 3 tập
df_original['word_count'] = df_original['text'].apply(lambda x: len(str(x).split()))
df_tuan['word_count'] = df_tuan['text'].apply(lambda x: len(str(x).split()))
df_vnfd['word_count'] = df_vnfd['text'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (df, name) in enumerate(zip([df_original, df_tuan, df_vnfd], ['Gốc', 'ViFN', 'VFND'])):
    sns.kdeplot(data=df, x='word_count', hue='label', fill=True, ax=axes[i], 
                palette=['#2ecc71', '#e74c3c'], common_norm=False)
    axes[i].set_xscale('log') # Dùng log-scale để nhìn rõ hơn
    axes[i].set_title(f'Phân phối chiều dài bài viết - {name}')
    axes[i].set_xlabel('Số lượng từ (Log scale)')

plt.tight_layout()
plt.savefig(CHART_DIR + '04_text_length_kde.png', bbox_inches='tight')
plt.show()

## 5. Chỉ số tương tác (Engagement)
*(Từ bước này trở đi, chúng ta tập trung phân tích sâu bộ dữ liệu Gốc `df_original`)*

**Giải thích chi tiết:** Các bài đăng tin giả thường sử dụng thủ thuật clickbait (câu view) để kích động cảm xúc, dẫn đến lượng Share và Comment cao bất thường so với Like. Ta sẽ vẽ Boxplot trên thang đo Logarit để loại bỏ nhiễu từ các bài đăng viral.

In [ ]:
# Chỉ lấy các cột tương tác có sẵn trong tập gốc
features_engagement = ['num_like', 'num_share', 'num_cmt']
# Cộng thêm 1 để tránh lỗi log(0)
for col in features_engagement:
    df_original[f'{col}_log'] = np.log1p(df_original[col])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(features_engagement):
    sns.boxplot(data=df_original, x='label', y=f'{col}_log', ax=axes[i], palette=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'Phân phối {col} (Log scale)')
    axes[i].set_xticklabels(['0 (Thật)', '1 (Giả)'])

plt.tight_layout()
plt.savefig(CHART_DIR + '05_engagement_boxplot.png', bbox_inches='tight')
plt.show()

## 6. Phân tích thời gian & Đặc trưng bài viết
Tin giả thường được tung ra vào các khung giờ nhất định (ví dụ ban đêm khi kiểm duyệt lỏng lẻo) và lạm dụng Emoji/URL để lừa đảo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Đặc trưng thời gian (Giờ đăng bài)
if 'hour' in df_original.columns:
    sns.histplot(data=df_original, x='hour', hue='label', multiple="dodge", bins=24, 
                 ax=axes[0], palette=['#2ecc71', '#e74c3c'])
    axes[0].set_title('Tần suất đăng bài theo Giờ trong ngày')

# Đặc trưng Emoji
if 'num_emoji' in df_original.columns:
    df_original['num_emoji_log'] = np.log1p(df_original['num_emoji'])
    sns.kdeplot(data=df_original, x='num_emoji_log', hue='label', fill=True, 
                ax=axes[1], palette=['#2ecc71', '#e74c3c'], common_norm=False)
    axes[1].set_title('Lượng sử dụng Emoji (Log scale)')

plt.tight_layout()
plt.savefig(CHART_DIR + '06_07_time_emoji.png', bbox_inches='tight')
plt.show()

## 7. Phân tích User (Chuyên sâu: Bằng chứng Rò rỉ Dữ liệu - Target Leakage)

Rò rỉ dữ liệu (Data Leakage) xảy ra khi mô hình được học từ những thông tin mà trong thực tế (lúc suy luận) nó không thể biết được. 
Trong bộ dữ liệu của chúng ta có các cột đếm tích lũy của người dùng: `num_real` (số bài thật đã đăng) và `num_fake` (số bài giả đã đăng).

In [ ]:
# Lọc ra các bài đăng ĐẦU TIÊN của mỗi người dùng
first_posts = df_original[df_original['num_post'] == 1]

# Một bài đăng đầu tiên đúng chuẩn (chưa biết tương lai) thì num_real và num_fake phải = 0
no_leakage_cases = first_posts[(first_posts['num_real'] == 0) & (first_posts['num_fake'] == 0)]
leakage_cases = first_posts[(first_posts['num_real'] != 0) | (first_posts['num_fake'] != 0)]

print(f"Tổng số bản ghi là bài đăng đầu tiên của user (num_post = 1): {len(first_posts)}")
print(f"Số bản ghi đúng chuẩn (num_real=0 & num_fake=0): {len(no_leakage_cases)}")
print(f"Số bản ghi BỊ RÒ RỈ DỮ LIỆU: {len(leakage_cases)}")

# Chứng minh bằng cách in ra 5 mẫu bị rò rỉ
print("\n--- Bằng chứng Rò rỉ dữ liệu (Nhìn vào Label và Num_fake/Num_real) ---")
display(leakage_cases[['user', 'label', 'num_post', 'num_real', 'num_fake', 'post_ratio']].head())

## 8. Ma trận tương quan (Correlation Matrix)
**Giải thích chi tiết:** Kiểm tra Đa cộng tuyến (Multicollinearity). Nếu 2 đặc trưng tương quan quá mạnh với nhau (ví dụ `num_like` và `num_cmt`), chúng sẽ cung cấp lượng thông tin trùng lặp, làm giảm tính ổn định của các mô hình tuyến tính (như Logistic Regression).

In [ ]:
# Chọn các cột số (numeric) để tính tương quan
cols_to_drop = ['num_real', 'num_fake', 'post_ratio'] # Các cột rò rỉ
numeric_cols = df_original.select_dtypes(include=[np.number]).columns.drop(cols_to_drop, errors='ignore')

corr_matrix = df_original[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Ma trận tương quan giữa các đặc trưng (Đã loại bỏ cột Leakage)')
plt.savefig(CHART_DIR + '09_correlation_matrix.png', bbox_inches='tight')
plt.show()

## 9. Phân tích Hình ảnh
Kiểm tra xem tỷ lệ bài viết có chứa hình ảnh (`has_image`) có ảnh hưởng đến khả năng đó là tin giả hay không.

In [ ]:
if 'has_image' in df_original.columns:
    plt.figure(figsize=(6, 5))
    sns.countplot(data=df_original, x='has_image', hue='label', palette=['#2ecc71', '#e74c3c'])
    plt.title('Mối liên hệ giữa Việc đính kèm Hình ảnh và Tin giả')
    plt.legend(['0 (Thật)', '1 (Giả)'])
    plt.savefig(CHART_DIR + '10_image_analysis.png', bbox_inches='tight')
    plt.show()
else:
    print("Dữ liệu không có cột 'has_image'")

## 10. Kết luận & Khuyến nghị cho Model

Dựa trên phân tích toàn diện, tôi đề xuất quy trình kỹ thuật cho bước Huấn luyện Mô hình (Training) như sau:

1. **Khắc phục Data Leakage (Bắt buộc):**
   * Các cột `num_real`, `num_fake` và `post_ratio` chứa trực tiếp thông tin nhãn của bài viết hiện tại. Nếu đưa vào mô hình (như LightGBM hay Logistic Regression), độ chính xác sẽ cao ảo (lên tới 99%) nhưng hoàn toàn vô dụng trong thực tế. **Phải Drop các cột này.**

2. **Xử lý Mất cân bằng lớp (Class Imbalance):**
   * Các biểu đồ phân phối cho thấy Tin thật chiếm đa số áp đảo so với Tin giả trên cả 3 tập dữ liệu.
   * **Khuyến nghị:** Áp dụng kỹ thuật Over-sampling (SMOTE) cho tập huấn luyện, hoặc sử dụng tham số `class_weight='balanced'` trong Logistic Regression / `scale_pos_weight` trong LightGBM.

3. **Chọn lọc Đặc trưng (Feature Selection):**
   * Nên giữ lại các đặc trưng đếm (Count features) có sự phân tách tốt: `num_emoji`, `word_count`, `num_like`, `num_share`.
   * Đối với đặc trưng Thời gian (`hour`, `month`), cần chuyển đổi bằng kỹ thuật Sin/Cos encoding (bởi vì giờ 23:00 và 01:00 mang ý nghĩa rất gần nhau trong thực tế).